In [181]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, IntSlider, FloatSlider, fixed, interactive_output, VBox, Output
DATA_DIR = "/home/tim-external/ros_ws/src/fsregistration/plotting_results/2d/data"
print('Imports OK')

Imports OK


# 2D SOFFT Registration Analysis

Interactive notebook for analyzing 2D SOFFT registration results. Use the sliders to browse rotation candidates and correlation surfaces.

In [182]:
def _detect_delim(line):
    if '\t' in line:
        return '\t'
    elif ',' in line:
        return ','
    return None


def load_csv(filename):
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        print(f'File not found: {filepath}')
        return None

    with open(filepath, 'r') as f:
        first_line = f.readline().strip()

    if not first_line:
        return None

    delim = _detect_delim(first_line)
    first_field = first_line.split(delim)[0] if delim else first_line.split()[0]
    is_header = not _is_numeric(first_field)

    if is_header:
        lines = []
        with open(filepath, 'r') as f2:
            next(f2)  # skip header
            for line in f2:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                # Skip the metadata row (6 columns) and section headers
                parts = line.split(delim) if delim else line.split()
                if len(parts) == 6 and _is_numeric(parts[0]):
                    continue  # skip metadata row like "128\t255\t..."
                if parts[0] in ('N', 'angleIndex'):
                    continue
                lines.append(line)
        if lines:
            d = _detect_delim(lines[0]) if delim is None else delim
            return np.loadtxt(lines, delimiter=d)
        return None

    # No header - use detected delimiter
    d = _detect_delim(first_line) if delim is None else delim
    return np.loadtxt(filepath, delimiter=d)


def _is_numeric(s):
    try:
        float(s)
        return True
    except ValueError:
        return False


In [183]:
# --- Load all data ---
print('Loading data...')

# Input data
input_voxel1 = load_csv('voxelDataFFTW1.csv')
input_voxel2 = load_csv('voxelDataFFTW2.csv')
mag_fftw1 = load_csv('magnitudeFFTW1.csv')
phase_fftw1 = load_csv('phaseFFTW1.csv')
mag_fftw2 = load_csv('magnitudeFFTW2.csv')
phase_fftw2 = load_csv('phaseFFTW2.csv')
resampled1 = load_csv('resampledVoxel1.csv')
resampled2 = load_csv('resampledVoxel2.csv')

# Rotation data
rotation_corr = load_csv('rotationCorrelation1D.csv')
rotation_peaks = load_csv('rotationPeaks.csv')
if rotation_peaks is not None:
    rotation_peaks = np.atleast_2d(rotation_peaks)

# Metadata from dataForReadIn.csv
meta_filepath = os.path.join(DATA_DIR, 'dataForReadIn.csv')
data_meta = None
angle_data = None
if os.path.exists(meta_filepath):
    with open(meta_filepath, 'r') as f:
        lines = f.readlines()
    # Metadata row is line 1 (0-indexed): 128, 255, 0.1, 0.01, 17, 0
    if len(lines) > 1:
        meta_parts = lines[1].strip().split('\t')
        data_meta = {
            'N': int(meta_parts[0]),
            'correlationN': int(meta_parts[1]),
            'cellSize': float(meta_parts[2]),
            'potentialNecessaryForPeak': float(meta_parts[3]),
            'numAngles': int(meta_parts[4]),
            'numTotalSolutions': int(meta_parts[5]) if len(meta_parts) > 5 else 0
        }
    # Per-angle section starts after "angleIndex\tangle\tnumTranslations"
    angle_data = []
    for j, line in enumerate(lines):
        if 'angleIndex' in line:
            for k in range(j + 1, len(lines)):
                parts = lines[k].strip().split('\t')
                if len(parts) >= 3:
                    angle_data.append({
                        'angleIndex': int(parts[0]),
                        'angle': float(parts[1]),
                        'numTranslations': int(parts[2])
                    })
            break

# Infer N
if input_voxel1 is not None and input_voxel1.ndim == 1:
    N = int(round(input_voxel1.size ** 0.5))
    print(f'Inferred N = {N}')
else:
    N = 128
    print(f'Using default N = {N}')

correlationN = 2 * N - 1
print(f'correlationN = {correlationN}')

# Reshape 2D data
if input_voxel1 is not None:
    input_voxel1 = input_voxel1.reshape((N, N))
    input_voxel2 = input_voxel2.reshape((N, N))
    mag_fftw1 = mag_fftw1.reshape((N, N))
    phase_fftw1 = phase_fftw1.reshape((N, N))
    mag_fftw2 = mag_fftw2.reshape((N, N))
    phase_fftw2 = phase_fftw2.reshape((N, N))
    resampled1 = resampled1.reshape((N, N))
    resampled2 = resampled2.reshape((N, N))

print('All input data loaded.')
print(f'  Voxel shapes: ({N},{N})')
print(f'  Input1: min={input_voxel1.min():.4f}, max={input_voxel1.max():.4f}')
print(f'  Input2: min={input_voxel2.min():.4f}, max={input_voxel2.max():.4f}')
if data_meta:
    print(f'  Metadata: N={data_meta["N"]}, angles={data_meta["numAngles"]}, solutions={data_meta["numTotalSolutions"]}')
if angle_data:
    print(f'  Per-angle data: {len(angle_data)} angles')


Loading data...
Inferred N = 256
correlationN = 511
All input data loaded.
  Voxel shapes: (256,256)
  Input1: min=0.0000, max=0.5061
  Input2: min=0.0000, max=0.5049
  Metadata: N=256, angles=4, solutions=116
  Per-angle data: 4 angles


In [184]:
fig = make_subplots(rows=2, cols=2, 
    subplot_titles=('Voxel 1', 'Voxel 2', 'Magnitude FFT 1', 'Magnitude FFT 2'),
    specs=[[{'type': 'heatmap'}, {'type': 'heatmap'}],
           [{'type': 'heatmap'}, {'type': 'heatmap'}]])

fig.add_trace(go.Heatmap(z=input_voxel1, colorscale='Viridis', showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=input_voxel2, colorscale='Viridis'), row=1, col=2)
fig.add_trace(go.Heatmap(z=mag_fftw1, colorscale='Viridis', showscale=False), row=2, col=1)
fig.add_trace(go.Heatmap(z=mag_fftw2, colorscale='Viridis'), row=2, col=2)

fig.update_layout(height=900, width=900, title_text='Input Data & Spectra')
fig.update_xaxes(nticks=20)
fig.update_yaxes(nticks=20)
fig.update_yaxes(scaleanchor="x",  scaleratio=1, row=1, col=1)
fig.update_yaxes(scaleanchor="x2", scaleratio=1, row=1, col=2)
fig.update_yaxes(scaleanchor="x3", scaleratio=1, row=2, col=1)
fig.update_yaxes(scaleanchor="x4", scaleratio=1, row=2, col=2)
fig.show()

## Input Data Visualization

Side-by-side comparison of the two input frames and their FFT spectra.

## Resampled Fourier Magnitudes

The 2D Fourier magnitude of each image is projected onto the unit circle (S² in 2D).

Note: the CSV stores the resampled grid transposed (rows = azimuth φ, columns = polar θ), so the data is transposed here for display:
the horizontal axis is the azimuth φ (angle in the FFT plane, measured from the +x frequency axis, 0..2π) and the vertical axis is the polar angle θ (0..π, i.e. r·sin(θ) sampling radius in the FFT plane).


In [ ]:
# --- Resampled Fourier Magnitudes ---
fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Resampled Magnitude 1', 'Resampled Magnitude 2'))

fig.add_trace(
    go.Heatmap(z=resampled1.T, colorscale='Viridis', showscale=False),
    row=1, col=1
)
fig.add_trace(
    go.Heatmap(z=resampled2.T, colorscale='Viridis'),
    row=1, col=2
)

fig.update_yaxes(scaleanchor='x', scaleratio=1, row=1, col=1)
fig.update_yaxes(scaleanchor='x2', scaleratio=1, row=1, col=2)
fig.update_layout(height=400, width=900, title_text='Resampled Fourier Magnitudes')
fig.update_xaxes(showticklabels=False, showgrid=False)
fig.update_yaxes(showticklabels=False, showgrid=False)
fig.show()


## Sphere Projection of Magnitude

The resampled 2D Fourier magnitude is projected onto the unit sphere S² using the same spherical coordinate parameterization as the 3D notebook. K3D mesh plots show both signals with a Jet colormap.

The azimuth on the sphere corresponds to φ (rows of the CSV), the polar angle to θ (columns of the CSV) — hence the attrs below are flattened without transposing.


In [ ]:
# --- Sphere Projection of Fourier Magnitude on S² (K3D) ---
import k3d
import numpy as np

N = resampled1.shape[0]
u = np.linspace(0, 2 * np.pi, N)
v = np.linspace(0, np.pi, N)

# Generate sphere vertices
vertices = np.zeros((N * N, 3))
for i in range(N):
    for j in range(N):
        idx = i * N + j
        vertices[idx, 0] = np.cos(u[i]) * np.sin(v[j])
        vertices[idx, 1] = np.sin(u[i]) * np.sin(v[j])
        vertices[idx, 2] = np.cos(v[j])

# Generate triangle indices (two triangles per quad)
indices = []
for i in range(N - 1):
    for j in range(N - 1):
        v0 = i * N + j
        v1 = (i + 1) * N + j
        v2 = (i + 1) * N + (j + 1)
        v3 = i * N + (j + 1)
        indices.append([v0, v2, v1])
        indices.append([v0, v3, v2])
indices = np.array(indices, dtype=np.uint32)

# Per-vertex attributes from resampled magnitude (no transpose: rows=φ azimuth, cols=θ polar)
attr1 = resampled1.flatten()
attr2 = resampled2.flatten()

# Normalize to [0, 1] for color mapping
attr1_norm = attr1 / attr1.max()
attr2_norm = attr2 / attr2.max()

# Plot 1: Scan 1
plot1 = k3d.plot(name='Sphere 1', camera_mode='orbit')
mesh1 = k3d.mesh(
    vertices.astype(np.float32),
    indices,
    attribute=attr1_norm.astype(np.float32),
    color_map=k3d.colormaps.basic_color_maps.Jet,
    color_range=[0, 1],
    flat_shading=False,
)
plot1 += mesh1
plot1.display()
print(f"Sphere 1: {len(vertices)} vertices, magnitude range [{attr1.min():.4f}, {attr1.max():.4f}]")

# Plot 2: Scan 2
plot2 = k3d.plot(name='Sphere 2', camera_mode='orbit')
mesh2 = k3d.mesh(
    vertices.astype(np.float32),
    indices,
    attribute=attr2_norm.astype(np.float32),
    color_map=k3d.colormaps.basic_color_maps.Jet,
    color_range=[0, 1],
    flat_shading=False,
)
plot2 += mesh2
plot2.display()
print(f"Sphere 2: {len(vertices)} vertices, magnitude range [{attr2.min():.4f}, {attr2.max():.4f}]")


## Rotation Peaks

Detected rotation peaks (red x markers) overlaid on the correlation curve. Each peak shows the angle in radians and its index in the correlation array.

In [187]:
fig = go.Figure()

if rotation_corr is not None and rotation_corr.ndim == 2:
    indices = rotation_corr[:, 0]
    angles = rotation_corr[:, 1]
    correlations = rotation_corr[:, 2]
    fig.add_trace(go.Scatter(x=angles, y=correlations, mode='lines', name='Correlation', line=dict(width=2, color='steelblue')))
    fig.update_xaxes(title_text='Rotation Angle (rad)')
    fig.update_yaxes(title_text='Normalized Correlation')

if rotation_peaks is not None and rotation_peaks.ndim == 2:
    sort_idx = np.argsort(rotation_peaks[:, 0])
    sorted_peaks = rotation_peaks[sort_idx]
    peak_angles = sorted_peaks[:, 0]
    peak_heights = sorted_peaks[:, 1]
    level_potentials = sorted_peaks[:, 3]
    peak_indices = sorted_peaks[:, 4].astype(int)
    peak_text = [f'{a:.2f}rad (idx={idx}) lvl={l:.4f}' for a, idx, l in zip(peak_angles, peak_indices, level_potentials)]
    fig.add_trace(go.Scatter(x=peak_angles, y=peak_heights, mode='markers+text',
                             marker=dict(size=12, color='red', symbol='x', line=dict(width=2)),
                             text=peak_text,
                             textposition='top center',
                             name='Detected Peaks'))

fig.update_layout(height=450, width=900, title_text='Rotation Peaks on Correlation Curve')
fig.show()

## Translation Correlation per Rotation Peak

For each detected rotation peak, the 1D translation correlation curve is computed.
Use the slider to select a rotation peak and inspect its translation correlation.
Red x markers indicate detected translation peaks for that rotation.


In [188]:
# --- Parse potentialTransformation files to build per-rotation translation data ---
import glob as glob_mod
import numpy.linalg as la

cellSize = data_meta['cellSize'] if data_meta else 1.0

def parse_transformation_matrix(filepath):
    '''Parse a potentialTransformation*.csv file and return rotation angle, translation, correlation, persistenceValue.'''
    rows = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            try:
                rows.append([float(x) for x in parts])
            except ValueError:
                continue

    if len(rows) < 4:
        return None

    # First 4 rows = 4x4 homogeneous matrix
    matrix = np.array(rows[:4])
    R = matrix[:3, :3]
    t = matrix[:3, 3]

    # Extract rotation angle from 2D rotation matrix
    angle = np.arctan2(R[1, 0], R[0, 0])

    # Correlation score is on the line after the 4x4 matrix
    corr_score = None
    pers_value = None
    if len(rows) >= 5 and len(rows[4]) == 1:
        corr_score = rows[4][0]
    if len(rows) >= 6 and len(rows[5]) == 1:
        pers_value = rows[5][0]

    return {
        'angle': angle,
        'translation': t,
        'matrix': matrix,
        'correlation': corr_score,
        'persistenceValue': pers_value,
        'translation_voxel_x': -t[0] / cellSize + (correlationN // 2) if cellSize != 0 else 0,
        'translation_voxel_y': -t[1] / cellSize + (correlationN // 2) if cellSize != 0 else 0,
    }


def find_closest_angle_index(angle, angle_list):
    '''Find the index of the rotation angle closest to the given angle.'''
    min_dist = float('inf')
    best_idx = 0
    for i, a in enumerate(angle_list):
        # Proper wrap-around normalization to [-pi, pi]
        diff = (angle - a + np.pi) % (2*np.pi) - np.pi
        dist = abs(diff)
        if dist < min_dist:
            min_dist = dist
            best_idx = i
    return best_idx
# Load all transformation files
trans_files = sorted(glob_mod.glob(os.path.join(DATA_DIR, 'potentialTransformation*.csv')))
max_sol = data_meta['numTotalSolutions'] if data_meta else 999999
trans_files = [f for f in trans_files
               if 0 <= int(os.path.basename(f).replace('potentialTransformation','').replace('.csv','')) < max_sol]
print(f'Found {len(trans_files)} transformation files (filtered to current run: max {max_sol})')

# Build mapping: rotation_angle_index -> list of translation solutions
if data_meta and angle_data:
    angle_list = [a['angle'] for a in angle_data]
    rotation_to_translations = {i: [] for i in range(len(angle_list))}
    rotation_to_corr_curve = {i: [] for i in range(len(angle_list))}

    for fpath in trans_files:
        basename = os.path.basename(fpath)
        # Extract number from filename
        num_str = basename.replace('potentialTransformation', '').replace('.csv', '')
        try:
            trans_num = int(num_str)
        except ValueError:
            continue

        trans_data = parse_transformation_matrix(fpath)
        if trans_data is None:
            continue

        angle = trans_data['angle']
        idx = find_closest_angle_index(angle, angle_list)
        rotation_to_translations[idx].append({
            'num': trans_num,
            'angle': angle,
            'translation': trans_data['translation'],
            'correlation': trans_data['correlation'],
            'translation_voxel_x': trans_data.get('translation_voxel_x'),
            'translation_voxel_y': trans_data.get('translation_voxel_y'),
            'persistenceValue': trans_data.get('persistenceValue'),
        })

    # Sort each rotation's translations by correlation score (descending)
    for idx in rotation_to_translations:
        rotation_to_translations[idx].sort(key=lambda x: x['correlation'] if x['correlation'] is not None else 0, reverse=True)

    # Build correlation curves: for each rotation, collect correlation values
    for idx in rotation_to_translations:
        sol_list = rotation_to_translations[idx]
        rotation_to_corr_curve[idx] = [s['correlation'] if s['correlation'] is not None else 0 for s in sol_list]

    # Shared rotation order for consistent display across cells
    rotation_order = np.argsort(angle_list)

    print(f'Rotation angles with solutions: {sum(1 for v in rotation_to_translations.values() if v)}')
    for idx in rotation_order:
        if rotation_to_translations.get(idx):
            print(f'  Angle {angle_list[idx]:.4f} rad (idx={idx}): {len(rotation_to_translations[idx])} solutions')
else:
    rotation_order = np.array([], dtype=int)
    rotation_to_translations = {}
    rotation_to_corr_curve = {}
    print('No angle data available for translation correlation')


# --- Interactive translation correlation plot ---
from ipywidgets import IntSlider, HBox, VBox, Label

if len(rotation_order) > 0:
    display_order = [i for i in rotation_order if i in rotation_to_translations]
    num_rotation_peaks = len(display_order)
else:
    display_order = []
    num_rotation_peaks = 0

@interact(peak_idx=(0, num_rotation_peaks - 1 if num_rotation_peaks > 0 else 0, 1))
def plot_translation_correlation(peak_idx):
    orig_idx = display_order[peak_idx] if len(display_order) > 0 else peak_idx
    if orig_idx >= len(angle_list) or not rotation_to_translations.get(orig_idx):
        fig = go.Figure()
        fig.add_annotation(text=f'No solutions for rotation index {orig_idx}',
                         xref='paper', yref='paper', showarrow=False,
                         font=dict(size=20))
        fig.update_layout(height=400, width=900)
        return fig

    angle = angle_list[orig_idx]
    solutions = rotation_to_translations[orig_idx]
    corr_values = rotation_to_corr_curve[orig_idx]
    pers_values = [s.get("persistenceValue", 0) or 0 for s in solutions]

    trans_indices = list(range(len(solutions)))

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=trans_indices,
        y=corr_values,
        mode='lines+markers',
        name='Correlation',
        line=dict(width=2, color='steelblue'),
        marker=dict(size=6),
        hovertemplate="#%{x}: corr=%{y:.3f}<br>lvl=%{customdata:.4f}",
        customdata=pers_values,
    ))

    # Mark the best (highest correlation) as a peak
    if corr_values:
        best_idx = np.argmax(corr_values)
        best_val = corr_values[best_idx]
        best_pers = solutions[best_idx].get('persistenceValue', 0) or 0
        fig.add_trace(go.Scatter(
            x=[best_idx],
            y=[best_val],
            mode='markers+text',
            marker=dict(size=14, color='red', symbol='x', line=dict(width=2)),
            text=[f'PEAK (corr={best_val:.3f}, lvl={best_pers:.4f})'],
            textposition='top center',
            name='Best Translation Peak',
        ))

    fig.update_layout(
        height=400, width=900,
        title_text=f'Translation Correlation — Rotation Peak {peak_idx} (orig={orig_idx}, angle={angle:.4f} rad, {len(solutions)} solutions)',
        xaxis_title_text='Translation Solution Index',
        yaxis_title_text='Correlation Score',
    )
    return fig


# ---- interact_manual removed, using @interact decorator above ----


Found 116 transformation files (filtered to current run: max 116)
Rotation angles with solutions: 4
  Angle 1.5693 rad (idx=0): 57 solutions
  Angle 2.0034 rad (idx=3): 13 solutions
  Angle 4.7062 rad (idx=1): 11 solutions
  Angle 6.2786 rad (idx=2): 35 solutions


interactive(children=(IntSlider(value=1, description='peak_idx', max=3), Output()), _dom_classes=('widget-inte…

## 2D Translation Correlation Surface

The 2D correlation 'hills' for the selected rotation angle.
Red 'x' markers indicate the detected translation peaks overlaid on the surface.


In [189]:
# --- Use shared angle-sorted order for consistent display ---
if len(rotation_order) > 0:
    display_order_corr = [i for i in rotation_order if i in rotation_to_translations]
    max_peak_idx = max(0, len(display_order_corr) - 1)
else:
    display_order_corr = []
    max_peak_idx = 0

@interact(peak_idx=(0, max_peak_idx, 1))
def plot_2d_correlation_surface(peak_idx):
    orig_idx = display_order_corr[peak_idx] if len(display_order_corr) > 0 else peak_idx
    corr_2d = load_csv(os.path.join(DATA_DIR, f'translationCorrelation2D_angle{orig_idx}.csv'))
    if corr_2d is None:
        return go.Figure()
    h, w = corr_2d.shape

    # --- Get translation peaks for this angle ---
    solutions = rotation_to_translations.get(orig_idx, [])
    peak_x, peak_y, peak_z, peak_corr, peak_pv = [], [], [], [], []
    for s in solutions:
        vx = s.get('translation_voxel_x')
        vy = s.get('translation_voxel_y')
        c = s.get('correlation')
        if vx is not None and vy is not None:
            ix = int(np.clip(np.round(vx), 0, w - 1))
            iy = int(np.clip(np.round(vy), 0, h - 1))
            peak_x.append(vx)
            peak_y.append(vy)
            peak_z.append(corr_2d[iy, ix])
            peak_corr.append(c if c is not None else 0)
            pv = s.get("persistenceValue")
            peak_pv.append(pv if pv is not None else 0)

    angle_val = angle_data[orig_idx]["angle"] if angle_data and orig_idx < len(angle_data) else 0.0

    fig = make_subplots(rows=1, cols=2,
        specs=[[{"type": "heatmap"}, {"type": "scene"}]],
        subplot_titles=(f"Correlation Heatmap — Peak {peak_idx} (orig={orig_idx})",
                         f"3D Surface — Peak {peak_idx} (orig={orig_idx})"))

    # Left: 2D heatmap with peak markers
    fig.add_trace(go.Heatmap(
        z=corr_2d, colorscale="Viridis", showscale=False
    ), row=1, col=1)
    if peak_x:
        fig.add_trace(go.Scatter(
            x=peak_x, y=peak_y, mode="markers",
            marker=dict(size=8, color="red", symbol="x", line=dict(width=2)),
            text=[f"cor={c:.3f} lvl={p:.4f}" for c, p in zip(peak_corr, peak_pv)],
            textposition="top center"
        ), row=1, col=1)

    # Right: 3D surface with peak markers
    fig.add_trace(go.Surface(
        z=corr_2d, colorscale="Viridis", showscale=True,
        colorbar_title="Correlation"
    ), row=1, col=2)
    if peak_x:
        fig.add_trace(go.Scatter3d(
            x=peak_x, y=peak_y, z=peak_z, mode="markers",
            marker=dict(size=4, color="red", symbol="cross",
                        line=dict(width=1, color="darkred")),
            text=[f"cor={c:.3f} lvl={p:.4f}" for c, p in zip(peak_corr, peak_pv)],
        ), row=1, col=2)

    fig.update_layout(
        height=600, width=1000,
        title_text=f"2D Translation Correlation — Peak {peak_idx} (orig={orig_idx}, angle={angle_val:.4f} rad, {len(solutions)} solutions)"
    )
    fig.update_xaxes(title_text="X Index", row=1, col=1)
    fig.update_yaxes(title_text="Y Index", row=1, col=1)
    fig.update_yaxes(scaleanchor='x', scaleratio=1, row=1, col=1)
    fig.update_scenes(
        xaxis_title="X", yaxis_title="Y", zaxis_title="Correlation",
        aspectmode="cube", row=1, col=2
    )
    return fig


interactive(children=(IntSlider(value=1, description='peak_idx', max=3), Output()), _dom_classes=('widget-inte…

## Blended Registration Results

For each rotation peak and each translation solution within that rotation,
the second voxel is transformed and blended with the first voxel.
Use the sliders to browse through rotation peaks and their translation solutions.
The blended image shows how well each candidate solution aligns the two scans.


In [190]:
from scipy.ndimage import affine_transform
import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact


def blend_images(voxel1, voxel2_transformed, alpha=0.5):
    '''Blend two images with given alpha.'''
    return (1 - alpha) * voxel1 + alpha * voxel2_transformed


# --- Pre-compute blended images for all solutions ---
print('Computing blended images...')
blended_images = {i: [] for i in range(len(angle_list)) if i in rotation_to_translations}
blended_info = {i: [] for i in range(len(angle_list)) if i in rotation_to_translations}
transformed_images = {i: [] for i in range(len(angle_list)) if i in rotation_to_translations}

for idx in blended_images:
    solutions = rotation_to_translations[idx]
    angle = angle_list[idx]
    print(f'  Processing rotation peak {idx} (angle={angle:.4f} rad, {len(solutions)} solutions)...')

    for sol in solutions:
        matrix = sol.get('matrix')
        if matrix is None:
            matrix = np.eye(4)
            R2d = np.array([[np.cos(sol['angle']), -np.sin(sol['angle'])],
                            [np.sin(sol['angle']),  np.cos(sol['angle'])]])
            matrix[:2, :2] = R2d
            matrix[:2, 3] = sol.get('translation', np.array([0, 0]))[:2]
        try:
            c = (N - 1) / 2.0
            R = matrix[:2, :2]
            t = matrix[:2, 3].copy() / cellSize
            t_rc = np.array([t[1], t[0]])
            c_rc = np.array([c, c])
            offset = c_rc - R.T @ c_rc + t_rc

            transformed = affine_transform(
                input_voxel2, R.T, offset=offset,
                output_shape=input_voxel2.shape,
                mode='constant', cval=0.0, order=1)

            blended = blend_images(input_voxel1, transformed, alpha=0.5)
            blended_images[idx].append(blended)
            transformed_images[idx].append(transformed)
            blended_info[idx].append({
                'sol_num': sol['num'],
                'correlation': sol.get('correlation'),
                'translation': sol.get('translation'),
                'translation_voxel_x': sol.get('translation_voxel_x'),
                'translation_voxel_y': sol.get('translation_voxel_y'),
            })
        except Exception as e:
            blended_images[idx].append(None)
            transformed_images[idx].append(None)
            blended_info[idx].append({
                'sol_num': sol['num'],
                'error': str(e),
            })

print('Done computing blended images.')

total = sum(len(v) for v in blended_images.values())
print(f'Total blended images: {total}')


# --- Sort translations within each rotation by norm (closest first) ---
for idx in blended_images:
    sols = rotation_to_translations[idx]
    sort_idx = np.argsort([np.linalg.norm(s['translation'][:2]) for s in sols])
    blended_images[idx] = [blended_images[idx][i] for i in sort_idx]
    transformed_images[idx] = [transformed_images[idx][i] for i in sort_idx]
    blended_info[idx] = [blended_info[idx][i] for i in sort_idx]

# --- Sort rotation peaks by angle (0 -> 2pi) ---
display_order = [i for i in rotation_order if i in blended_images]
max_rot = max(0, len(display_order) - 1)
max_trans = max(len(v) for v in blended_images.values()) - 1 if blended_images else 0


@interact(rot_idx=(0, max_rot, 1), trans_idx=(0, max_trans, 1))
def show_blended_image(rot_idx=0, trans_idx=0):
    orig_idx = display_order[rot_idx]

    if orig_idx not in blended_images or not blended_images[orig_idx]:
        fig = go.Figure()
        fig.add_annotation(text='No blended images available',
                         xref='paper', yref='paper', showarrow=False,
                         font=dict(size=20))
        fig.update_layout(height=600, width=600)
        fig.show()
        return

    n_solutions = len(blended_images[orig_idx])
    trans_idx = min(trans_idx, n_solutions - 1)

    transformed = transformed_images[orig_idx][trans_idx]
    info = blended_info[orig_idx][trans_idx]

    if transformed is None:
        fig = go.Figure()
        fig.add_annotation(text=f'Error for solution {info.get("sol_num", "?")}: {info.get("error", "unknown")}',
                         xref='paper', yref='paper', showarrow=False,
                         font=dict(size=16))
        fig.update_layout(height=600, width=600)
        fig.show()
        return

    v1 = (input_voxel1 - input_voxel1.min()) / (input_voxel1.max() - input_voxel1.min() + 1e-10)
    v2 = (transformed - transformed.min()) / (transformed.max() - transformed.min() + 1e-10)

    pad = 0
    canvas_h = N + 2 * pad
    canvas_w = N + 2 * pad
    v1_canvas = np.zeros((canvas_h, canvas_w))
    v1_canvas[pad:pad+N, pad:pad+N] = v1
    v2_canvas = np.zeros((canvas_h, canvas_w))
    v2_canvas[pad:pad+N, pad:pad+N] = v2

    rgb = np.ones((canvas_h, canvas_w, 3)) * 255
    # v1 (reference) -> blue, v2 (transformed) -> green, bg -> white
    rgb[:,:,0] -= np.maximum(v1_canvas, v2_canvas) * 255
    rgb[:,:,1] -= v1_canvas * 255
    rgb[:,:,2] -= v2_canvas * 255

    fig = go.Figure()
    fig.add_trace(go.Image(z=rgb))

    fig.add_annotation(
        x=0.95, y=0.95,
        xref='paper', yref='paper',
        text=f"sol #{info.get('sol_num', '?')}",
        showarrow=False,
        font=dict(size=20, color='white'),
        bgcolor='rgba(0,0,0,0.5)',
        xanchor='right', yanchor='top'
    )

    rot_angle = angle_list[orig_idx] if orig_idx < len(angle_list) else 0
    corr = info.get('correlation')
    corr_str = f', corr={corr:.2f}' if corr is not None else ''
    t = info.get('translation', np.array([0, 0]))
    vx = info.get('translation_voxel_x')
    vy = info.get('translation_voxel_y')
    shift_str = f'<br>Shift: dx={t[0]:.2f}, dy={t[1]:.2f}'
    if vx is not None and vy is not None:
        shift_str += f' (voxel: x={vx:.1f}, y={vy:.1f})'
    title = (f'Rotation Peak {rot_idx} (orig={orig_idx}, angle={rot_angle:.4f} rad) '
             f'<br>Translation Solution {trans_idx} '
             f'(sol_num={info.get("sol_num", "?")}{corr_str})'
             f'{shift_str}')

    fig.update_layout(height=800, width=800, title_text=title)
    fig.update_xaxes(showticklabels=False, showgrid=False)
    fig.update_yaxes(showticklabels=False, showgrid=False)
    fig.update_yaxes(scaleanchor='x', scaleratio=1)
    fig.show()


Computing blended images...
  Processing rotation peak 0 (angle=1.5693 rad, 57 solutions)...
  Processing rotation peak 1 (angle=4.7062 rad, 11 solutions)...
  Processing rotation peak 2 (angle=6.2786 rad, 35 solutions)...
  Processing rotation peak 3 (angle=2.0034 rad, 13 solutions)...
Done computing blended images.
Total blended images: 116


interactive(children=(IntSlider(value=0, description='rot_idx', max=3), IntSlider(value=0, description='trans_…

## Translation Solutions per Rotation Angle

Bar chart showing how many translation solutions were detected for each candidate rotation angle.

In [191]:
if data_meta and angle_data and len(rotation_order) > 0:
    fig = go.Figure()
    
    angles = [angle_data[i]['angle'] for i in rotation_order
              if i < len(angle_data)]
    num_trans = [angle_data[i]['numTranslations'] for i in rotation_order
                 if i < len(angle_data)]
    angle_indices = [angle_data[i]['angleIndex'] for i in rotation_order
                    if i < len(angle_data)]
    
    fig.add_trace(go.Bar(
        x=angles,
        y=num_trans,
        text=[f'idx={idx}, n={n}' for idx, n in zip(angle_indices, num_trans)],
        textposition='outside',
        marker_color='steelblue',
        marker_line_color='darkblue',
        marker_line_width=1.5,
        name='Translation Solutions'
    ))
    
    fig.update_xaxes(title_text='Rotation Angle (rad)')
    fig.update_yaxes(title_text='Number of Translation Peaks')
    fig.update_layout(height=450, width=900, title_text='Translation Solutions per Rotation Angle')
    fig.show()
else:
    print('No dataForReadIn.csv data available')


## Transform Summary

Overview of all detected transforms across all rotation angles.

In [192]:
# Load dataForReadIn.csv for summary
if data_meta and angle_data:
    print(f'Number of rotation angles: {data_meta["numAngles"]}')
    print(f'Total solutions: {data_meta["numTotalSolutions"]}')
    print()
    print('Rotation Angle Summary (sorted by angle):')
    print(f'{"Idx":>4} {"Angle (rad)":>12} {"# Trans":>8}')
    print('-' * 28)
    for i in rotation_order:
        if i < len(angle_data):
            a = angle_data[i]
            print(f'{a["angleIndex"]:>4} {a["angle"]:>12.4f} {a["numTranslations"]:>8}')
    print()
    
    # Bar chart sorted by angle
    sorted_angle_data = [angle_data[i] for i in rotation_order if i < len(angle_data)]
    fig, ax = plt.subplots(figsize=(12, 4))
    indices = [a['angleIndex'] for a in sorted_angle_data]
    num_trans = [a['numTranslations'] for a in sorted_angle_data]
    ax.bar(range(len(indices)), num_trans, color='steelblue', edgecolor='black')
    ax.set_xticks(range(len(indices)))
    ax.set_xticklabels([f'{i}\n{a["angle"]:.2f}' for i, a in zip(indices, sorted_angle_data)], fontsize=8)
    ax.set_title('Number of Translation Solutions per Rotation Angle (sorted by angle)')
    ax.set_xlabel('Angle Index / Rotation Angle (rad)')
    ax.set_ylabel('Number of Translation Peaks')
    fig.tight_layout()
    
    plt.show()
elif not data_meta:
    print('dataForReadIn.csv not found or could not be parsed')
else:
    print('dataForReadIn.csv loaded but angle data is empty')


Number of rotation angles: 4
Total solutions: 116

Rotation Angle Summary (sorted by angle):
 Idx  Angle (rad)  # Trans
----------------------------
   0       1.5693       57
   3       2.0034       13
   1       4.7062       11
   2       6.2786       35



/tmp/ipykernel_11691/3442807815.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
